# Trace Each LLM Answer Back to Its Media File

In this video, I'm going to do two things. First, send marketplace photos to Gemini and keep each answer on the same row as the photo it read and its listing. Then follow each Gemini call in Logfire, Pydantic's observability tool.

*Show pydantic-ai issue #2139 in the browser.*

Once your photos are bytes in a message, how do you tell which one came from where? In pydantic-ai issue 2139, a tool couldn't tell which image to act on, because there wasn't a good way to tag `BinaryContent` with where it came from.

*Show pydantic-ai issue #2635.*

Pydantic AI added an `identifier` for that, and issue 2635 asked for one on URLs too, because a presigned URL changes each time you make one. Both are in Pydantic AI today.

## Start Jupyter with your Gemini and Logfire keys set

The notebook is in the `trace-answers-to-media` folder of the repo linked below. Off camera, I started Jupyter there with `uv run jupyter lab`, after setting my Gemini and Logfire keys in the shell.

In [ ]:
!uv sync

## In a Pydantic AI message, a photo is bytes and an identifier

Here's one of our photos the way Pydantic AI sends it to a model: the bytes and a media type.

In [ ]:
import httpx
from pydantic_ai import BinaryContent

BASE = 'https://raw.githubusercontent.com/apreshill/pixeltable-marketplace/main/samples/'

part = BinaryContent(httpx.get(BASE + 'backpack-1.jpg').content, media_type='image/jpeg')
part.identifier

By default, its identifier is a six-character hash of the bytes, and you can pass your own string instead. Keeping track of which listing each photo belongs to, and what the model said about it, is still up to your code.

## Each answer is a column on its photo's row

I'll start from an empty `trace_demo` directory.

In [ ]:
import pixeltable as pxt

# Start fresh: drop the tables from any earlier run.
pxt.drop_dir('trace_demo', force=True, if_not_exists='ignore')

`ListingDetails` is a plain Pydantic model for Gemini's answer. `PROMPT` is the one sentence we send with each photo.

In [ ]:
!sed -n '9,20p' schema.py

In Pixeltable, a multimodal backend, you declare a table as a Python class. In `listings`, you insert a listing ID and a seller with each photo. `photo` is an `Image` column, so the URL you insert is read as an image.

In [ ]:
!sed -n '26,29p' schema.py

`response` is a computed column: for every new row, Pixeltable sends Gemini the prompt and that row's photo, and asks for JSON in the shape of `ListingDetails`.

In [ ]:
!sed -n '31,36p' schema.py

`details` checks that the answer has each field of `ListingDetails`, with the right type, before the row is stored. It doesn't check `Literal` values, like the list of categories.

`pxt schema diff` shows the plan, and `pxt schema update` creates the table.

In [ ]:
!pxt schema diff schema.py trace_demo

In [ ]:
!pxt schema update schema.py trace_demo -f

## Four listings go in, and Gemini answers each photo during the insert

Each photo is a URL from the marketplace repo. Pixeltable downloads it and sends it to Gemini during the insert, one paid call per photo. These four take about 15 seconds.

In [ ]:
listings = pxt.get_table('trace_demo.listings')
listings.insert(
    [
        {'listing_id': 101, 'seller': 'lee', 'photo': BASE + 'backpack-1.jpg'},
        {'listing_id': 102, 'seller': 'omar', 'photo': BASE + 'bankers-lamp-2.jpg'},
        {'listing_id': 103, 'seller': 'eva', 'photo': BASE + 'cruiser-bike-3.jpg'},
        {'listing_id': 104, 'seller': 'raj', 'photo': BASE + 'desk-lamp-1.jpg'},
    ]
)

That's four rows and four Gemini calls, with no errors.

## Each answer comes back next to the photo it read

Each answer is on the same row as its photo and its listing.

In [ ]:
listings.select(
    listings.listing_id, listings.seller, listings.photo, listings.details.title, listings.details.condition
).order_by(listings.listing_id).collect()

## A `where` on the answer leads back to the photo and its URL

To check an answer, you look at the photo the model read. Here, a `where` on Gemini's category finds the home listings, and `photo.fileurl` gives back the URL each photo came from.

In [ ]:
listings.where(listings.details.category == 'home').select(
    listings.photo, listings.photo.fileurl, listings.listing_id, listings.seller, listings.details.title
).collect()

Both lamps came back, each with the photo Gemini read and the URL we inserted.

## Logfire shows each Gemini call as its own span

The table says which photo each answer came from, but not how each Gemini call went, like how long it took. Logfire shows that.

`logfire.configure` reads your `LOGFIRE_TOKEN`, and `PixeltableInstrumentor` sends Pixeltable's spans to Logfire. At `debug`, every Gemini call on every row gets its own span.

In [ ]:
import logfire
from opentelemetry.instrumentation.pixeltable import PixeltableInstrumentor
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

spans = InMemorySpanExporter()  # demo only: a copy of each span, to read in this notebook

logfire.configure(
    service_name='trace-answers-to-media',
    send_to_logfire='if-token-present',  # reads LOGFIRE_TOKEN
    console=False,
    additional_span_processors=[SimpleSpanProcessor(spans)],
)
PixeltableInstrumentor().instrument(span_level='debug')  # a span for each model call on each row

The in-memory exporter is only for this video. It keeps a copy of each span, so we can read them here too.

I'll post two more listings with one insert each, the way an app posts one listing per request. That way, each trace holds one photo's download and its Gemini call.

In [ ]:
for listing in [
    {'listing_id': 105, 'seller': 'maya', 'photo': BASE + 'film-camera-1.jpg'},
    {'listing_id': 106, 'seller': 'kofi', 'photo': BASE + 'guitar-1.jpg'},
]:
    listings.insert([listing])

Here are the download span and the Gemini span from each insert.

In [ ]:
for s in spans.get_finished_spans():
    if s.name in ('pixeltable.media.fetch', 'pixeltable.udf.generate_content'):
        trace_end = f'{s.context.trace_id:032x}'[-8:]  # Logfire's trace IDs start with a timestamp
        attrs = {k: v for k, v in s.attributes.items() if k in ('pxt.url', 'pxt.column')}
        print(f'trace ...{trace_end}', s.name, attrs, f'{(s.end_time - s.start_time) / 1e9:.1f} s')

Each insert is one trace. Its download span names the photo's URL, and its Gemini span names the column it filled, `response`, which tells the calls apart when a table has more than one model column.

In a multi-row insert, the trace doesn't say which call read which photo, and an uploaded photo has no download span.

*Switch to Logfire, filter to the `trace-answers-to-media` service, open the newest `pixeltable.insert` trace, and click its `pixeltable.media.fetch` span.*

Here's the guitar's insert: the download span's `pxt.url` is the guitar photo's URL. *Click the `pixeltable.udf.generate_content` span.* Its `pxt.column` is `response`, and its duration is how long Gemini took on that photo.

## Each answer stayed with its photo, and each Gemini call got its own span

Each Gemini answer went on the same row as the photo it read, and a `where` on the answer led back to the photo and its URL. In Logfire, each insert's trace held the photo's URL and its Gemini call, named by its column.

The notebook and schema file are in the `trace-answers-to-media` folder of the repo linked below. If this was useful, a star on the Pixeltable GitHub repo helps other developers find it.